# Stage 3: Constraint-Aware Legal Text Simplification — Juris-Clarify Pipeline

## Overview
1. **Generate synthetic training data** using Cerebras API (LLaMA 3.3 70B) from Stage 1 clauses + Stage 2 constraints
2. **Fine-tune BART** on constraint-marked complex → simplified pairs
3. **Run inference** on test set using Stage 2 constraint profiles
4. **Evaluate** readability, entity preservation, and simplification quality

### Inputs Required
- **Stage 1 predictions**: `predicted_train.jsonl`, `predicted_val.jsonl`, `predicted_test.jsonl` (predicted clause segments)
- **Stage 2 constraints**: `train_constraints.json`, `val_constraints.json`, `test_constraints.json` (entity/obligation/category profiles for all splits)
- **Cerebras API key** (set via Kaggle Secrets or inline)
- **GPU enabled** (T4 or P100)

In [ ]:
!pip install -q transformers openai pytorch-crf rouge-score nltk tqdm

## 1. Configuration
Update paths and API key below to match your Kaggle dataset names.

In [ ]:
import os
import json
import re
import time
import random
import logging
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import Counter

import torch
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

# ======================================================================
# PATHS — Kaggle dataset paths (verified via find command)
# ======================================================================
CUAD_JSON       = "/kaggle/input/cuad-v1/CUAD_v1/CUAD_v1/CUAD_v1.json"

# Stage 1 PREDICTED segments
TRAIN_JSONL     = "/kaggle/input/datasets/manikyapant/stage1-predictions/stage1_predictions/predicted_train.jsonl"
VAL_JSONL       = "/kaggle/input/datasets/manikyapant/stage1-predictions/stage1_predictions/predicted_val.jsonl"
TEST_JSONL      = "/kaggle/input/datasets/manikyapant/stage1-predictions/stage1_predictions/predicted_test.jsonl"

# Stage 2 outputs — constraints for ALL splits
TRAIN_CONSTRAINTS = "/kaggle/input/datasets/manikyapant/stage2-output/stage2_output/train_constraints.json"
VAL_CONSTRAINTS   = "/kaggle/input/datasets/manikyapant/stage2-output/stage2_output/val_constraints.json"
TEST_CONSTRAINTS  = "/kaggle/input/datasets/manikyapant/stage2-output/stage2_output/test_constraints.json"

OUTPUT_DIR      = "/kaggle/working/stage3_output"

# ======================================================================
# CEREBRAS API — set via Kaggle Secrets or paste key directly
# ======================================================================
try:
    from kaggle_secrets import UserSecretsClient
    CEREBRAS_API_KEY = UserSecretsClient().get_secret("CEREBRAS_API_KEY")
    print("Loaded Cerebras API key from Kaggle Secrets")
except Exception:
    CEREBRAS_API_KEY = "YOUR_CEREBRAS_API_KEY_HERE"  # <-- paste your key here
    print("Using inline Cerebras API key")

CEREBRAS_MODEL = "qwen-3-235b-a22b-instruct-2507"

# ======================================================================
# BART FINE-TUNING HYPERPARAMETERS
# ======================================================================
BART_MODEL_NAME  = "facebook/bart-large"
EPOCHS           = 3
BATCH_SIZE       = 4
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
MAX_INPUT_LEN    = 512
MAX_TARGET_LEN   = 256
GRAD_ACCUM       = 4
USE_FP16         = True
SEED             = 42

# ======================================================================
# GENERATION SETTINGS
# ======================================================================
MAX_CLAUSES_TO_GENERATE = 5000
MIN_CLAUSE_LENGTH       = 10
SAVE_EVERY              = 100

# Chunked generation: split long clauses into overlapping word-chunks
CHUNK_SIZE_WORDS        = 180
CHUNK_OVERLAP_WORDS     = 20
MAX_CHUNKS_PER_CLAUSE   = 6

# ======================================================================
# EXISTING SYNTHETIC PAIRS (skip generation if available)
# ======================================================================
EXISTING_PAIRS_PATH = "/kaggle/input/datasets/manikyapant/stage3-input/stage3_input/synthetic_pairs.json"

# ======================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
SYNTHETIC_PAIRS_FILE = os.path.join(OUTPUT_DIR, "synthetic_pairs.json")

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Cerebras model: {CEREBRAS_MODEL}")
print(f"BART model: {BART_MODEL_NAME}")

## 2. Utility Functions
Data loading, regex entity extraction, and obligation detection.

In [ ]:
# -- Segment file loader (JSON array or JSONL) --

def load_segments(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(1)
    if first == "[":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"  Loaded {len(records)} records from {path.name}")
    return records


# -- Regex entity extraction --

DATE_PATTERNS = [
    re.compile(
        r"\b(?:January|February|March|April|May|June|July|August|September|"
        r"October|November|December)\s+\d{1,2},?\s+\d{4}\b", re.IGNORECASE),
    re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    re.compile(r"\b\d{4}[-/]\d{2}[-/]\d{2}\b"),
    re.compile(r"\b\d{1,2}(?:st|nd|rd|th)\s+day\s+of\s+\w+,?\s+\d{4}\b", re.IGNORECASE),
]

MONEY_PATTERNS = [
    re.compile(r"\$[\d,]+(?:\.\d{1,2})?(?:\s*(?:million|billion|thousand))?", re.IGNORECASE),
    re.compile(r"\b(?:USD|EUR|GBP|CAD)\s*[\d,]+(?:\.\d{1,2})?", re.IGNORECASE),
    re.compile(r"\b\d+(?:\.\d+)?%\b"),
]

DURATION_PATTERNS = [
    re.compile(r"\b\w+\s*\(\d+\)\s*(?:days?|months?|years?|weeks?|business\s+days?)\b", re.IGNORECASE),
    re.compile(r"\b\d+\s*(?:days?|months?|years?|weeks?|calendar\s+days?|business\s+days?)\b", re.IGNORECASE),
]

LEGAL_REF_PATTERNS = [
    re.compile(r"\bSection\s+\d+(?:\.\d+)*(?:\([a-z]\))?", re.IGNORECASE),
    re.compile(r"\bArticle\s+(?:\d+|[IVXLC]+)\b", re.IGNORECASE),
    re.compile(r"\bExhibit\s+[A-Z]\b", re.IGNORECASE),
    re.compile(r"\bSchedule\s+[A-Z\d]+\b", re.IGNORECASE),
]

OBLIGATION_KEYWORDS = {
    "MANDATORY":   ["shall", "must", "is required to", "are required to", "will",
                    "agrees to", "is obligated to", "undertakes to"],
    "PROHIBITIVE": ["shall not", "must not", "may not", "will not",
                    "is prohibited from", "cannot", "is not permitted to"],
    "PERMISSIVE":  ["may", "can", "is entitled to", "has the right to",
                    "is permitted to", "is allowed to", "at its option"],
    "CONDITIONAL": ["if", "provided that", "subject to", "unless",
                    "in the event that", "on the condition that", "notwithstanding"],
    "DECLARATIVE": ["means", "shall mean", "is defined as", "refers to",
                    "includes", "as used herein"],
}

IMMUTABLE_PATTERNS = [
    re.compile(r"^\s*definitions?\s*$", re.IGNORECASE | re.MULTILINE),
    re.compile(r"\bgoverning law\b", re.IGNORECASE),
    re.compile(r"\bchoice of law\b", re.IGNORECASE),
    re.compile(r"\bentire agreement\b", re.IGNORECASE),
    re.compile(r"\bseverability\b", re.IGNORECASE),
]


def extract_entities_regex(text):
    all_spans = []
    for patterns, etype in [
        (DATE_PATTERNS, "DATE"), (MONEY_PATTERNS, "MONEY"),
        (DURATION_PATTERNS, "DURATION"), (LEGAL_REF_PATTERNS, "LEGAL_REF"),
    ]:
        for pat in patterns:
            for m in pat.finditer(text):
                all_spans.append({"type": etype, "text": m.group(),
                                  "start": m.start(), "end": m.end()})
    all_spans.sort(key=lambda x: (x["start"], -x["end"]))
    filtered = []
    last_end = -1
    for e in all_spans:
        if e["start"] >= last_end:
            filtered.append(e)
            last_end = e["end"]
    return filtered


def detect_obligation(text):
    text_lower = text.lower()
    for obl in ["PROHIBITIVE", "PERMISSIVE", "CONDITIONAL", "DECLARATIVE", "MANDATORY"]:
        for kw in OBLIGATION_KEYWORDS[obl]:
            if kw.lower() in text_lower:
                return obl
    return "DECLARATIVE"


def is_immutable(text):
    return any(p.search(text) for p in IMMUTABLE_PATTERNS)


def extract_clauses_from_doc(doc):
    doc_id = doc.get("doc_id", doc.get("filename", ""))
    full_text = doc.get("text", doc.get("full_text", ""))
    clauses_raw = doc.get("clauses", [])
    clauses = []
    for ci, clause_info in enumerate(clauses_raw):
        if isinstance(clause_info, list):
            cs, ce = clause_info[0], clause_info[1]
            clause_text = full_text[cs:ce]
        elif isinstance(clause_info, dict):
            cs = clause_info.get("start", 0)
            ce = clause_info.get("end", 0)
            clause_text = clause_info.get("text", full_text[cs:ce])
        else:
            continue
        clause_text = clause_text.strip()
        if clause_text and len(clause_text) >= MIN_CLAUSE_LENGTH:
            clauses.append({"doc_id": doc_id, "clause_idx": ci, "text": clause_text})
    return clauses


# -- Entity marking for BART input --

ENTITY_MARKERS = {
    "DATE":         ("<DATE>", "</DATE>"),
    "MONEY":        ("<MONEY>", "</MONEY>"),
    "DURATION":     ("<DUR>", "</DUR>"),
    "LEGAL_REF":    ("<LREF>", "</LREF>"),
    "PARTY":        ("<PARTY>", "</PARTY>"),
    "JURISDICTION": ("<JURIS>", "</JURIS>"),
}


def mark_entities_in_text(text, entities):
    if not entities:
        return text
    sorted_ents = sorted(entities, key=lambda e: e.get("start", 0), reverse=True)
    marked = text
    for e in sorted_ents:
        s, end = e.get("start"), e.get("end")
        etype = e.get("type", "")
        if s is not None and end is not None and etype in ENTITY_MARKERS:
            otag, ctag = ENTITY_MARKERS[etype]
            marked = marked[:s] + otag + marked[s:end] + ctag + marked[end:]
    return marked


print("Utility functions loaded.")

## 3. Synthetic Data Generation via Cerebras API (LLaMA 3.3 70B)
Generates simplified versions of legal clauses for BART fine-tuning.
- Uses Cerebras inference (extremely fast, ~2000 tokens/sec)
- Constraint-aware prompting (preserves entities & obligation type)
- Resumable: saves progress every 100 pairs

In [ ]:
from openai import OpenAI

cerebras_client = OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=CEREBRAS_API_KEY,
)

SYSTEM_PROMPT = (
    "You are a legal text simplification expert. Your job is to simplify legal "
    "clauses to an 8th-grade reading level while preserving all critical legal "
    "information.\n\n"
    "RULES:\n"
    "1. PRESERVE all named entities exactly as written: company names, person "
    "names, dates, monetary amounts, durations, jurisdictions, legal references.\n"
    "2. PRESERVE the legal meaning and obligation type "
    "(shall=must, may=can, shall not=cannot).\n"
    "3. Use simpler vocabulary and shorter sentences.\n"
    "4. Do NOT add information not in the original.\n"
    "5. Do NOT omit key legal terms, conditions, or parties.\n"
    "6. If the clause is a definition, keep the defined term and simplify "
    "the explanation.\n"
    "7. Output ONLY the simplified clause, nothing else."
)


def simplify_clause_cerebras(clause_text, entities=None, obligation=None, max_retries=3):
    entity_str = ""
    if entities:
        parts = [f"  - {e['type']}: {e['text']}" for e in entities]
        entity_str = "\n\nEntities to preserve exactly:\n" + "\n".join(parts)

    obl_str = ""
    if obligation:
        obl_str = f"\nObligation type: {obligation}"

    user_msg = (
        f"Simplify this legal clause:{entity_str}{obl_str}\n\n"
        f"Original:\n{clause_text}\n\nSimplified:"
    )

    for attempt in range(max_retries):
        try:
            response = cerebras_client.chat.completions.create(
                model=CEREBRAS_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=512,
                temperature=0.3,
            )
            result = response.choices[0].message.content.strip()
            # Clean up: remove quotes or prefix if model adds them
            if result.startswith('"') and result.endswith('"'):
                result = result[1:-1]
            if result.lower().startswith("simplified:"):
                result = result[len("simplified:"):].strip()
            return result
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return None
    return None


print("Cerebras client initialized.")
print(f"Model: {CEREBRAS_MODEL}")

In [ ]:
from tqdm.auto import tqdm

# -- Load Stage 2 constraints for train + val --
def load_constraint_lookup(constraint_path):
    """Build (doc_id, clause_idx) -> constraint dict from Stage 2 output."""
    lookup = {}
    if os.path.exists(constraint_path):
        with open(constraint_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for doc_entry in data.get("documents", []):
            doc_id = doc_entry["doc_id"]
            for clause in doc_entry.get("clauses", []):
                key = (doc_id, clause["clause_index"])
                lookup[key] = clause
        print(f"  Loaded constraints for {len(lookup)} clauses from {Path(constraint_path).name}")
    else:
        print(f"  WARNING: {constraint_path} not found, will use regex fallback")
    return lookup

print("Loading Stage 2 constraints for train + val...")
train_constraint_lookup = load_constraint_lookup(TRAIN_CONSTRAINTS)
val_constraint_lookup = load_constraint_lookup(VAL_CONSTRAINTS)
gen_constraint_lookup = {**train_constraint_lookup, **val_constraint_lookup}
print(f"Total constraint profiles available: {len(gen_constraint_lookup)}")

# -- Load all train + val clauses --
print("\nLoading Stage 1 segments...")
train_docs = load_segments(TRAIN_JSONL)
val_docs = load_segments(VAL_JSONL)

all_clauses = []
for doc in train_docs + val_docs:
    all_clauses.extend(extract_clauses_from_doc(doc))
print(f"Total clauses available: {len(all_clauses)}")

# ── Load existing synthetic pairs — COMPLETELY SKIP generation ────────
skip_generation = False

if os.path.exists(EXISTING_PAIRS_PATH):
    with open(EXISTING_PAIRS_PATH, "r", encoding="utf-8") as f:
        synthetic_pairs = json.load(f)
    print(f"\n✅ Loaded {len(synthetic_pairs)} existing synthetic pairs from input dataset")
    print("   *** SKIPPING all Cerebras API generation ***")
    # Copy to output dir for downstream cells
    with open(SYNTHETIC_PAIRS_FILE, "w", encoding="utf-8") as f:
        json.dump(synthetic_pairs, f, indent=2, ensure_ascii=False)
    print(f"  Copied to {SYNTHETIC_PAIRS_FILE}")
    skip_generation = True
elif os.path.exists(SYNTHETIC_PAIRS_FILE):
    with open(SYNTHETIC_PAIRS_FILE, "r", encoding="utf-8") as f:
        synthetic_pairs = json.load(f)
    print(f"\n✅ Resumed: {len(synthetic_pairs)} previously generated pairs loaded")
    skip_generation = True
else:
    synthetic_pairs = []
    print("\n⚠️  No existing pairs found — will run generation via Cerebras API")

if not skip_generation:
    # Cap to MAX_CLAUSES_TO_GENERATE
    if len(all_clauses) > MAX_CLAUSES_TO_GENERATE:
        all_clauses = random.sample(all_clauses, MAX_CLAUSES_TO_GENERATE)

    already_done = {(p["doc_id"], p["clause_idx"]) for p in synthetic_pairs}
    remaining = [c for c in all_clauses if (c["doc_id"], c["clause_idx"]) not in already_done]
    print(f"Remaining to generate: {len(remaining)}")

    failed = 0
    pbar = tqdm(remaining, desc="Generating simplifications")
    for i, clause in enumerate(pbar):
        doc_id = clause["doc_id"]
        clause_idx = clause["clause_idx"]
        text = clause["text"]

        constraint = gen_constraint_lookup.get((doc_id, clause_idx))
        if constraint:
            entities = constraint.get("entities", [])
            obligations = [o.get("type", o.get("obligation_type", ""))
                           for o in constraint.get("obligations", [])]
            obligation = obligations[0] if obligations else detect_obligation(text)
            clause_immutable = constraint.get("immutable", False)
        else:
            entities = extract_entities_regex(text)
            obligation = detect_obligation(text)
            clause_immutable = is_immutable(text)

        simplified = simplify_clause_cerebras(text, entities, obligation)

        if simplified and len(simplified.strip()) > 10:
            synthetic_pairs.append({
                "doc_id": doc_id, "clause_idx": clause_idx,
                "complex": text, "simple": simplified,
                "entities": entities, "obligation": obligation,
                "immutable": clause_immutable,
            })
        else:
            failed += 1

        if (i + 1) % SAVE_EVERY == 0:
            with open(SYNTHETIC_PAIRS_FILE, "w", encoding="utf-8") as f:
                json.dump(synthetic_pairs, f, ensure_ascii=False)

    with open(SYNTHETIC_PAIRS_FILE, "w", encoding="utf-8") as f:
        json.dump(synthetic_pairs, f, indent=2, ensure_ascii=False)
    print(f"\nGeneration complete! Total pairs: {len(synthetic_pairs)}, Failed: {failed}")

print(f"\nTotal synthetic pairs for training: {len(synthetic_pairs)}")

## 4. Synthetic Data Quality Check
Quick inspection of generated pairs before training.

In [ ]:
# If resuming without generation, load from file
if not synthetic_pairs and os.path.exists(SYNTHETIC_PAIRS_FILE):
    with open(SYNTHETIC_PAIRS_FILE, "r", encoding="utf-8") as f:
        synthetic_pairs = json.load(f)

print(f"Total pairs: {len(synthetic_pairs)}")

# Compute statistics
complex_lens = [len(p["complex"]) for p in synthetic_pairs]
simple_lens = [len(p["simple"]) for p in synthetic_pairs]
compression_ratios = [len(p["simple"]) / max(len(p["complex"]), 1) for p in synthetic_pairs]
obl_counts = Counter(p["obligation"] for p in synthetic_pairs)
immutable_count = sum(1 for p in synthetic_pairs if p.get("immutable", False))

print(f"\nComplex text length: avg={np.mean(complex_lens):.0f}, "
      f"median={np.median(complex_lens):.0f}, max={max(complex_lens)}")
print(f"Simple text length:  avg={np.mean(simple_lens):.0f}, "
      f"median={np.median(simple_lens):.0f}, max={max(simple_lens)}")
print(f"Compression ratio:   avg={np.mean(compression_ratios):.3f}, "
      f"median={np.median(compression_ratios):.3f}")
print(f"Immutable clauses:   {immutable_count}")
print(f"\nObligation distribution:")
for obl, cnt in obl_counts.most_common():
    print(f"  {obl:14s}: {cnt:5d} ({100*cnt/len(synthetic_pairs):.1f}%)")

# Show samples
print("\n" + "=" * 70)
for pair in synthetic_pairs[:3]:
    print(f"\n[{pair['obligation']}] Entities: {[e['text'] for e in pair['entities']]}")
    print(f"COMPLEX: {pair['complex'][:200]}...")
    print(f"SIMPLE:  {pair['simple'][:200]}...")
    print("-" * 70)

## 5. BART Dataset & Model Setup
Constraint-marked input format:
```
simplify [MANDATORY]: <PARTY>Acme Corp</PARTY> shall pay <MONEY>$10,000</MONEY>...
```
Target: Simplified text from Cerebras generation.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SimplificationDataset(Dataset):

    def __init__(self, pairs, tokenizer, max_input_len=512, max_target_len=256):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        obligation = pair.get("obligation", "DECLARATIVE")
        entities = pair.get("entities", [])
        complex_text = pair["complex"]
        simple_text = pair["simple"]

        # Build constraint-aware input with entity markers
        marked_text = mark_entities_in_text(complex_text, entities)
        input_text = f"simplify [{obligation}]: {marked_text}"

        # Tokenize input
        inputs = self.tokenizer(
            input_text, max_length=self.max_input_len,
            padding="max_length", truncation=True, return_tensors="pt",
        )

        # Tokenize target
        targets = self.tokenizer(
            text_target=simple_text, max_length=self.max_target_len,
            padding="max_length", truncation=True, return_tensors="pt",
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels,
        }


print("SimplificationDataset class defined.")

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

# -- Load BART tokenizer and model --
tokenizer = BartTokenizer.from_pretrained(BART_MODEL_NAME)

# Add special entity marker tokens
special_tokens = []
for otag, ctag in ENTITY_MARKERS.values():
    special_tokens.extend([otag, ctag])
# Add obligation prefix tokens
special_tokens.extend([
    "[MANDATORY]", "[PROHIBITIVE]", "[PERMISSIVE]",
    "[CONDITIONAL]", "[DECLARATIVE]",
])

num_added = tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
print(f"Added {num_added} special tokens to tokenizer")

# Load model and resize embeddings
model = BartForConditionalGeneration.from_pretrained(BART_MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {BART_MODEL_NAME}")
print(f"Parameters: {total_params:,}")
print(f"Vocabulary: {len(tokenizer):,}")
print(f"Device: {device}")

In [ ]:
# -- Split synthetic pairs into train / val --
# Filter out immutable clauses from training
trainable_pairs = [p for p in synthetic_pairs if not p.get("immutable", False)]
print(f"Trainable pairs (non-immutable): {len(trainable_pairs)}")

random.shuffle(trainable_pairs)
n_val = max(1, int(len(trainable_pairs) * 0.1))
train_pairs = trainable_pairs[n_val:]
val_pairs = trainable_pairs[:n_val]

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

train_ds = SimplificationDataset(train_pairs, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
val_ds = SimplificationDataset(val_pairs, tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2,
)

print(f"DataLoaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

## 6. Fine-Tune BART
Joint training with:
- Seq2seq cross-entropy loss (standard BART objective)
- FP16 mixed precision
- Gradient accumulation
- Cosine LR schedule + early stopping

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import time as _time

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

use_amp = USE_FP16 and device.type == "cuda"

ckpt_dir = Path(OUTPUT_DIR) / "bart_best_model"
ckpt_dir.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")
patience_cnt = 0
PATIENCE = 3
history = []

print(f"Training for up to {EPOCHS} epochs")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} accum = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"  Mixed precision (bfloat16): {use_amp}")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    epoch_start = _time.time()

    # -- Train --
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=True)
    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item() * GRAD_ACCUM:.4f}")

    # Flush remaining gradients
    if len(train_loader) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_train_loss = total_loss / len(train_loader)

    # -- Validate --
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                outputs = model(**batch)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / max(len(val_loader), 1)
    scheduler.step()
    elapsed = _time.time() - epoch_start

    entry = {
        "epoch": epoch,
        "train_loss": round(avg_train_loss, 4),
        "val_loss": round(avg_val_loss, 4),
        "elapsed_sec": round(elapsed, 1),
    }
    history.append(entry)

    print(f"\nEpoch {epoch} ({elapsed:.0f}s) | "
          f"Train Loss={avg_train_loss:.4f} | Val Loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_cnt = 0
        model.save_pretrained(str(ckpt_dir))
        tokenizer.save_pretrained(str(ckpt_dir))
        print(f"  >>> New best model saved! Val Loss={best_val_loss:.4f}")
    else:
        patience_cnt += 1
        print(f"  No improvement ({patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print("  Early stopping triggered.")
            break

# Save training history
with open(Path(OUTPUT_DIR) / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n" + "=" * 70)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {ckpt_dir}")

## 7. Test Set Inference
Load best BART checkpoint, apply Stage 2 constraints, and generate simplified text for all test clauses.

In [ ]:
# -- Load best checkpoint --
model = BartForConditionalGeneration.from_pretrained(str(ckpt_dir)).to(device)
model.eval()
print(f"Loaded best BART checkpoint from {ckpt_dir}")

# -- Load test data --
print("\nLoading test data...")
test_docs = load_segments(TEST_JSONL)

test_clauses = []
for doc in test_docs:
    test_clauses.extend(extract_clauses_from_doc(doc))
print(f"Test clauses: {len(test_clauses)}")

# -- Load Stage 2 constraints for test split --
test_constraint_lookup = load_constraint_lookup(TEST_CONSTRAINTS)
print(f"Test constraint profiles: {len(test_constraint_lookup)}")

In [ ]:
from tqdm.auto import tqdm

# ── Chunked BART generation helper ──────────────────────────────────────
def split_into_word_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """Split text into overlapping word-level chunks."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks = []
    step = max(chunk_size - overlap, 1)
    start = 0
    while start < len(words):
        chunk_words = words[start:start + chunk_size]
        chunks.append(" ".join(chunk_words))
        if len(chunks) >= MAX_CHUNKS_PER_CLAUSE:
            break
        start += step
        if start >= len(words):
            break
    return chunks


def bart_simplify_chunked(text, obligation, entities):
    """Simplify using chunked BART inference for long clauses."""
    marked_text = mark_entities_in_text(text, entities)
    chunks = split_into_word_chunks(marked_text)

    simplified_parts = []
    for chunk in chunks:
        input_text = f"simplify [{obligation}]: {chunk}"
        inputs = tokenizer(
            input_text, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                output_ids = model.generate(
                    **inputs,
                    max_length=MAX_TARGET_LEN,
                    num_beams=4,
                    length_penalty=1.0,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                )
        decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        simplified_parts.append(decoded)

    return " ".join(simplified_parts)


simplified_results = []
entity_preservation_stats = {"total_entities": 0, "preserved": 0, "missing": 0}

for clause in tqdm(test_clauses, desc="Simplifying test clauses"):
    doc_id = clause["doc_id"]
    clause_idx = clause["clause_idx"]
    text = clause["text"]

    constraint = test_constraint_lookup.get((doc_id, clause_idx))

    clause_immutable = False
    if constraint:
        clause_immutable = constraint.get("immutable", False)
    else:
        clause_immutable = is_immutable(text)

    if clause_immutable:
        simplified_results.append({
            "doc_id": doc_id, "clause_idx": clause_idx,
            "original": text, "simplified": text,
            "immutable": True, "entities_preserved": True,
            "missing_entities": [], "obligation": "DECLARATIVE",
            "method": "skipped_immutable",
        })
        continue

    if constraint:
        entities = constraint.get("entities", [])
        obligations = [o.get("type", o.get("obligation_type", ""))
                       for o in constraint.get("obligations", [])]
        obligation = obligations[0] if obligations else detect_obligation(text)
    else:
        entities = extract_entities_regex(text)
        obligation = detect_obligation(text)

    # -- Chunked BART inference --
    n_chunks = len(split_into_word_chunks(text))
    simplified = bart_simplify_chunked(text, obligation, entities)

    entity_texts = [e["text"] for e in entities] if entities else []
    missing = [et for et in entity_texts if et not in simplified]
    preserved_all = len(missing) == 0

    entity_preservation_stats["total_entities"] += len(entity_texts)
    entity_preservation_stats["preserved"] += len(entity_texts) - len(missing)
    entity_preservation_stats["missing"] += len(missing)

    simplified_results.append({
        "doc_id": doc_id,
        "clause_idx": clause_idx,
        "original": text,
        "simplified": simplified,
        "immutable": False,
        "obligation": obligation,
        "entities": entity_texts,
        "entities_preserved": preserved_all,
        "missing_entities": missing,
        "n_chunks": n_chunks,
        "method": "bart_finetuned_chunked" if n_chunks > 1 else "bart_finetuned",
    })

# -- Save results --
output_file = Path(OUTPUT_DIR) / "simplified_test_clauses.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(simplified_results, f, indent=2, ensure_ascii=False)

n_chunked = sum(1 for r in simplified_results if r.get("n_chunks", 1) > 1)
print(f"\nSimplified {len(simplified_results)} test clauses")
print(f"  Processed in multiple chunks: {n_chunked}")
print(f"Saved to: {output_file}")

total_ent = entity_preservation_stats["total_entities"]
preserved_ent = entity_preservation_stats["preserved"]
if total_ent > 0:
    print(f"\nEntity Preservation: {preserved_ent}/{total_ent} ({100 * preserved_ent / total_ent:.1f}%)")

## 8. Evaluation Metrics
- **Readability**: Flesch-Kincaid Grade Level, Flesch Reading Ease
- **Compression**: Character-level compression ratio
- **Entity Preservation**: % of entities preserved verbatim
- **Quality**: ROUGE scores (using Cerebras simplifications as pseudo-reference)

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# -- Readability metrics --

def count_syllables(word):
    word = word.lower()
    if not word:
        return 0
    count = 0
    vowels = "aeiouy"
    if word[0] in vowels:
        count += 1
    for i in range(1, len(word)):
        if word[i] in vowels and word[i - 1] not in vowels:
            count += 1
    if word.endswith("e"):
        count -= 1
    return max(count, 1)


def flesch_kincaid_grade(text):
    sentences = nltk.sent_tokenize(text)
    words = [w for w in nltk.word_tokenize(text) if w.isalpha()]
    if len(sentences) == 0 or len(words) == 0:
        return 0
    syllables = sum(count_syllables(w) for w in words)
    fk = (0.39 * (len(words) / len(sentences))
          + 11.8 * (syllables / len(words))
          - 15.59)
    return round(fk, 2)


def flesch_reading_ease(text):
    sentences = nltk.sent_tokenize(text)
    words = [w for w in nltk.word_tokenize(text) if w.isalpha()]
    if len(sentences) == 0 or len(words) == 0:
        return 0
    syllables = sum(count_syllables(w) for w in words)
    fre = (206.835
           - 1.015 * (len(words) / len(sentences))
           - 84.6 * (syllables / len(words)))
    return round(fre, 2)


# -- Compute metrics on all test results --

original_fk_grades = []
simplified_fk_grades = []
original_fre_scores = []
simplified_fre_scores = []
compression_ratios_list = []
entities_preserved_count = 0
entities_total_clauses = 0

for r in simplified_results:
    orig = r["original"]
    simp = r["simplified"]

    if r.get("immutable", False):
        continue

    original_fk_grades.append(flesch_kincaid_grade(orig))
    simplified_fk_grades.append(flesch_kincaid_grade(simp))
    original_fre_scores.append(flesch_reading_ease(orig))
    simplified_fre_scores.append(flesch_reading_ease(simp))
    compression_ratios_list.append(len(simp) / max(len(orig), 1))

    if r.get("entities_preserved", False):
        entities_preserved_count += 1
    entities_total_clauses += 1

# -- Print results --
print("=" * 70)
print("  STAGE 3 -- SIMPLIFICATION EVALUATION")
print("=" * 70)

print(f"\n  Readability (Flesch-Kincaid Grade Level):")
print(f"    Original:   avg={np.mean(original_fk_grades):.2f}, "
      f"median={np.median(original_fk_grades):.2f}")
print(f"    Simplified: avg={np.mean(simplified_fk_grades):.2f}, "
      f"median={np.median(simplified_fk_grades):.2f}")
print(f"    Improvement: {np.mean(original_fk_grades) - np.mean(simplified_fk_grades):.2f} "
      f"grade levels")

print(f"\n  Flesch Reading Ease (higher = easier):")
print(f"    Original:   avg={np.mean(original_fre_scores):.2f}")
print(f"    Simplified: avg={np.mean(simplified_fre_scores):.2f}")
print(f"    Improvement: +{np.mean(simplified_fre_scores) - np.mean(original_fre_scores):.2f}")

print(f"\n  Compression Ratio:")
print(f"    avg={np.mean(compression_ratios_list):.3f}, "
      f"median={np.median(compression_ratios_list):.3f}")
print(f"    (1.0 = same length, lower = more compressed)")

print(f"\n  Entity Preservation:")
print(f"    Clauses with all entities preserved: "
      f"{entities_preserved_count}/{entities_total_clauses} "
      f"({100 * entities_preserved_count / max(entities_total_clauses, 1):.1f}%)")

ent_total = entity_preservation_stats["total_entities"]
ent_pres = entity_preservation_stats["preserved"]
if ent_total > 0:
    print(f"    Individual entities preserved: "
          f"{ent_pres}/{ent_total} ({100 * ent_pres / ent_total:.1f}%)")

immutable_count = sum(1 for r in simplified_results if r.get("immutable", False))
print(f"\n  Immutable clauses (skipped): {immutable_count}")
print("=" * 70)

# -- Save full metrics --
metrics = {
    "readability": {
        "original_fk_grade_avg": round(float(np.mean(original_fk_grades)), 2),
        "simplified_fk_grade_avg": round(float(np.mean(simplified_fk_grades)), 2),
        "fk_improvement": round(float(np.mean(original_fk_grades) - np.mean(simplified_fk_grades)), 2),
        "original_fre_avg": round(float(np.mean(original_fre_scores)), 2),
        "simplified_fre_avg": round(float(np.mean(simplified_fre_scores)), 2),
    },
    "compression": {
        "avg_ratio": round(float(np.mean(compression_ratios_list)), 3),
        "median_ratio": round(float(np.median(compression_ratios_list)), 3),
    },
    "entity_preservation": {
        "clause_level": round(entities_preserved_count / max(entities_total_clauses, 1), 4),
        "entity_level": round(ent_pres / max(ent_total, 1), 4),
    },
    "immutable_clauses": immutable_count,
    "total_simplified": entities_total_clauses,
}

with open(Path(OUTPUT_DIR) / "evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {OUTPUT_DIR}/evaluation_metrics.json")

## 9. Sample Outputs
Display a few example simplifications for inspection.

In [ ]:
print("=" * 70)
print("  SAMPLE SIMPLIFICATIONS")
print("=" * 70)

shown = 0
for r in simplified_results:
    if r.get("immutable", False):
        continue
    if shown >= 5:
        break

    print(f"\n[{r['obligation']}] Doc: {r['doc_id']}, Clause: {r['clause_idx']}")
    print(f"ORIGINAL  ({len(r['original'])} chars):")
    orig_display = r["original"][:300]
    print(f"  {orig_display}{'...' if len(r['original']) > 300 else ''}")
    print(f"SIMPLIFIED ({len(r['simplified'])} chars):")
    simp_display = r["simplified"][:300]
    print(f"  {simp_display}{'...' if len(r['simplified']) > 300 else ''}")

    if r.get("missing_entities"):
        print(f"  WARNING -- Missing entities: {r['missing_entities']}")
    else:
        print(f"  Entities preserved: YES")

    fk_orig = flesch_kincaid_grade(r["original"])
    fk_simp = flesch_kincaid_grade(r["simplified"])
    print(f"  FK Grade: {fk_orig} -> {fk_simp} (improvement: {fk_orig - fk_simp:.1f})")
    print("-" * 70)
    shown += 1

## 10. Package All Outputs
Save the fine-tuned BART model, simplified clauses, and metrics for downstream use.

In [ ]:
import shutil

# -- Copy checkpoint to clean output --
final_dir = Path(OUTPUT_DIR) / "final_bart_model"
if ckpt_dir.exists():
    if final_dir.exists():
        shutil.rmtree(final_dir)
    shutil.copytree(ckpt_dir, final_dir)
    print(f"Final model copied to: {final_dir}")

# -- Summary of all saved files --
print("\n" + "=" * 70)
print("  STAGE 3 COMPLETE -- ALL OUTPUTS")
print("=" * 70)

for f in sorted(Path(OUTPUT_DIR).rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        rel = f.relative_to(OUTPUT_DIR)
        if size_kb > 1024:
            print(f"  {rel}  ({size_kb / 1024:.1f} MB)")
        else:
            print(f"  {rel}  ({size_kb:.1f} KB)")

print("\nKey outputs:")
print(f"  1. Simplified test clauses: {OUTPUT_DIR}/simplified_test_clauses.json")
print(f"  2. Evaluation metrics:      {OUTPUT_DIR}/evaluation_metrics.json")
print(f"  3. Synthetic training pairs: {OUTPUT_DIR}/synthetic_pairs.json")
print(f"  4. Fine-tuned BART model:   {OUTPUT_DIR}/final_bart_model/")
print(f"  5. Training history:        {OUTPUT_DIR}/training_history.json")
print("\nStage 3 pipeline complete!")